# 🔥 İleri Veri Mühendisliği — Bölüm 2: Apache Spark Uygulamalı Lab (Google Colab)

Bu not defteri, HTML rehberdeki konuları **çalışan kodla** pekiştirir. Her hücreyi sırayla çalıştırın (`Çalışma zamanı → Tümünü çalıştır` da olur).

| # | Konu |
|---|------|
| 0 | Kurulum (Java 17 + PySpark) ve Spark UI |
| 1 | RDD: transformation/action, lineage, narrow/wide, partition'lar |
| 2 | Sentetik veri seti üretimi (2M sipariş, 50K kullanıcı) — CSV vs Parquet |
| 3 | DataFrame API: filtre, toplama, join, pencere fonksiyonları |
| 4 | Spark SQL (CTE, window, Spark 4 pipe sözdizimi) |
| 5 | Catalyst: `explain`, predicate pushdown, partition pruning |
| 6 | Join stratejileri, AQE ve veri çarpıklığı (salting) |
| 7 | Cache / persist |
| 8 | Python UDF vs yerleşik fonksiyon vs Pandas UDF |
| 9 | Structured Streaming: pencere, watermark, `foreachBatch`, `availableNow` |
| 10 | MLlib: Pipeline + CrossValidator ile churn modeli |
| 11 | Executor boyutlandırma hesaplayıcısı |
| 12 | Alıştırmalar |

> 💡 Colab'ın ücretsiz katmanında ~2 çekirdek ve ~12 GB RAM vardır; veri boyutları buna göre seçildi. Tüm lab yaklaşık 5–8 dakikada çalışır.

## 0. Kurulum

In [ ]:
import os, sys, subprocess, time

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Spark 4.x Java 17+ ister. Colab'da Java 17'yi kurup JAVA_HOME'u ayarlıyoruz.
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless > /dev/null",
                   shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.4"], check=True)

# Python worker'larının not defteriyle AYNI Python yorumlayıcısını (ve pandas/pyarrow'u) kullanmasını garanti et
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

BASE = "/content/spark_lab" if IN_COLAB else os.path.abspath("spark_lab")
os.makedirs(BASE, exist_ok=True)
print("Colab mı?:", IN_COLAB, "| Çalışma dizini:", BASE)
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0])

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import *
import pyspark

spark = (SparkSession.builder
         .appName("IleriVeriMuhendisligi-SparkLab")
         .master("local[*]")                                  # tüm çekirdekler
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")         # küçük küme için 200 yerine 8
         .config("spark.sql.adaptive.enabled", "true")        # AQE (3.2+ varsayılan)
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("PySpark:", pyspark.__version__, "| Spark:", spark.version)
print("defaultParallelism (çekirdek):", sc.defaultParallelism)

### Spark UI
Job → Stage → Task akışını, shuffle miktarlarını ve DAG görselini görmek için Spark UI'ı açın. Aşağıdaki hücre Colab'da yeni bir sekmede açılan bir bağlantı üretir. Lab boyunca ara ara bu sekmeyi yenileyin.

In [ ]:
if IN_COLAB:
    from google.colab import output
    output.serve_kernel_port_as_window(4040, path="/jobs/")
else:
    print("Spark UI:", sc.uiWebUrl)

In [ ]:
# Yardımcılar: süre ölçme ve fiziksel planı metin olarak yakalama
import io, contextlib

class timer:
    def __init__(self, label): self.label = label
    def __enter__(self): self.t = time.time(); return self
    def __exit__(self, *a): print(f"⏱  {self.label}: {time.time()-self.t:.2f} sn")

def plan_text(df, mode="formatted"):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain(mode)
    return buf.getvalue()

def grep_plan(df, *keys, mode="formatted"):
    # Plandan yalnızca ilgili satırları göster
    for line in plan_text(df, mode).splitlines():
        if any(k in line for k in keys):
            print("  ", line.strip()[:170])

def run(df):
    # Sonucu hiçbir yere yazmadan tüm planı çalıştırır — performans ölçümü için ideal
    df.write.format("noop").mode("overwrite").save()

def du(path):
    total = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path) for f in fs)
    return total / 1024**2

## 1. RDD Temelleri

**Transformation**'lar tembeldir (lazy): sadece soy ağacına (lineage) bir adım eklerler. **Action** çağrılınca DAG oluşturulur, shuffle noktalarında **stage**'lere bölünür ve çalıştırılır.

In [ ]:
text = [
    "Hadoop veriyi bloklara böler ve replike eder",
    "Spark veriyi bellekte işler ve DAG ile optimize eder",
    "Spark Hadoop YARN üzerinde çalışabilir",
    "HDFS Spark ve Hive için ortak depolama katmanıdır",
] * 2500

rdd    = sc.parallelize(text, 4)
words  = rdd.flatMap(lambda l: l.lower().split())     # narrow
pairs  = words.map(lambda w: (w, 1))                  # narrow
counts = pairs.reduceByKey(lambda a, b: a + b)        # WIDE → shuffle → yeni stage
print("Buraya kadar hiçbir iş çalışmadı (lazy evaluation).")

top = counts.takeOrdered(8, key=lambda kv: -kv[1])    # ACTION → job başlar
for w, c in top:
    print(f"{w:12s} {c:>6}")

print("\nLineage (girinti değişimi = shuffle sınırı = yeni stage):")
print(counts.toDebugString().decode())

### Partition'lar, `repartition` ve `coalesce`

In [ ]:
nums = sc.parallelize(range(1, 21), 4)
print("Başlangıç (4 partition):", nums.glom().collect())
print("repartition(3)  → tam shuffle, dengeli :", nums.repartition(3).glom().collect())
print("coalesce(2)     → shuffle yok, birleştir:", nums.coalesce(2).glom().collect())

# mapPartitions: pahalı bir başlangıç işlemini (ör. DB bağlantısı) partition başına bir kez yapmak için
def per_partition(it):
    conn = "bağlantı-açıldı"          # kayıt başına değil, partition başına 1 kez
    yield (conn, sum(it))
print("mapPartitions:", nums.mapPartitions(per_partition).collect())

### `reduceByKey` vs `groupByKey`
`reduceByKey`, shuffle'dan **önce** her partition'da yerel toplama yapar (MapReduce'taki combiner gibi). `groupByKey` ise tüm değerleri ağ üzerinden taşır.

In [ ]:
import random
random.seed(42)
big = sc.parallelize([(random.randint(0, 99), 1) for _ in range(600_000)], 8)

with timer("reduceByKey"):
    r1 = big.reduceByKey(lambda a, b: a + b).collect()
with timer("groupByKey + sum"):
    r2 = big.groupByKey().mapValues(sum).collect()
print("Sonuçlar eşit mi?", sorted(r1) == sorted(r2))
print("👉 Spark UI → Stages sekmesinde iki işin 'Shuffle Write' miktarlarını karşılaştırın.")

### Paylaşılan değişkenler: Broadcast ve Accumulator

In [ ]:
city_lookup = {0: "İstanbul", 1: "Ankara", 2: "İzmir", 3: "Bursa"}
bc_cities   = sc.broadcast(city_lookup)       # her executor'a BİR KEZ gönderilir
missing     = sc.accumulator(0)               # task'lardan driver'a sayaç

def enrich(rec):
    uid, cid = rec
    name = bc_cities.value.get(cid)
    if name is None:
        missing.add(1)
    return (uid, name or "BİLİNMİYOR")

data = sc.parallelize([(i, i % 6) for i in range(30)], 3)
out = data.map(enrich).collect()
print(out[:8])
print("Eşleşmeyen kayıt:", missing.value)
print("⚠️ Accumulator bir transformation içinde güncellenirse, task yeniden denemelerinde iki kez sayılabilir.")

## 2. Sentetik Veri Seti

Gerçekçi bir e-ticaret senaryosu üretiyoruz. Siparişlerdeki `user_id` **bilerek çarpık** (skewed) üretiliyor: küçük ID'li kullanıcılar çok daha fazla sipariş veriyor. Bu, Bölüm 6'da işimize yarayacak.

In [ ]:
N_USERS, N_ORDERS = 50_000, 2_000_000
cities = ["İstanbul", "Ankara", "İzmir", "Bursa", "Antalya", "Konya", "Adana", "Trabzon"]
plans  = ["free", "basic", "pro"]

def pick(values, seed):
    arr = F.array(*[F.lit(v) for v in values])
    return F.element_at(arr, (F.rand(seed) * len(values)).cast("int") + 1)

users = (spark.range(N_USERS).withColumnRenamed("id", "user_id")
         .withColumn("city", pick(cities, 1))
         .withColumn("plan", pick(plans, 2))
         .withColumn("signup_date", F.date_sub(F.lit("2026-09-30").cast("date"), (F.rand(3) * 720).cast("int"))))

start = F.unix_timestamp(F.lit("2026-09-01 00:00:00"))
orders = (spark.range(N_ORDERS).withColumnRenamed("id", "order_id")
          .withColumn("user_id", (F.pow(F.rand(4), 6) * N_USERS).cast("long"))        # çarpık dağılım
          .withColumn("amount", F.round(F.rand(5) * 990 + 10, 2))
          .withColumn("status", F.when(F.rand(6) < 0.88, "COMPLETED")
                                 .when(F.rand(7) < 0.6, "CANCELLED").otherwise("REFUNDED"))
          .withColumn("created_at", F.timestamp_seconds(start + (F.rand(8) * 86400 * 30).cast("long")))
          .withColumn("dt", F.to_date("created_at")))

with timer("Yazma: users parquet"):
    users.write.mode("overwrite").parquet(f"{BASE}/users")
with timer("Yazma: orders CSV"):
    orders.write.mode("overwrite").option("header", True).csv(f"{BASE}/orders_csv")
with timer("Yazma: orders Parquet"):
    orders.write.mode("overwrite").parquet(f"{BASE}/orders_parquet")
with timer("Yazma: orders Parquet (dt'ye göre bölümlenmiş)"):
    orders.write.mode("overwrite").partitionBy("dt").parquet(f"{BASE}/orders_by_dt")

print(f"\nCSV boyutu     : {du(BASE + '/orders_csv'):7.1f} MB")
print(f"Parquet boyutu : {du(BASE + '/orders_parquet'):7.1f} MB  (sütunsal + sıkıştırılmış)")

### Şema çıkarımı (inferSchema) vs açık şema

In [ ]:
with timer("CSV + inferSchema (veri iki kez taranır)"):
    df_inf = spark.read.option("header", True).option("inferSchema", True).csv(f"{BASE}/orders_csv")
    df_inf.count()

schema = StructType([
    StructField("order_id", LongType()), StructField("user_id", LongType()),
    StructField("amount", DoubleType()), StructField("status", StringType()),
    StructField("created_at", TimestampType()), StructField("dt", DateType())])
with timer("CSV + açık şema"):
    spark.read.option("header", True).schema(schema).csv(f"{BASE}/orders_csv").count()
with timer("Parquet (şema dosyanın içinde)"):
    spark.read.parquet(f"{BASE}/orders_parquet").count()

orders = spark.read.parquet(f"{BASE}/orders_parquet")
users  = spark.read.parquet(f"{BASE}/users")
orders.printSchema()
orders.show(5, truncate=False)

## 3. DataFrame API

In [ ]:
completed = orders.filter(F.col("status") == "COMPLETED")

# Şehir bazında günlük ciro — join + groupBy
daily_city = (completed.join(users, "user_id")
              .groupBy("dt", "city")
              .agg(F.round(F.sum("amount"), 2).alias("revenue"),
                   F.countDistinct("user_id").alias("buyers"),
                   F.count("*").alias("n_orders")))

# Pencere fonksiyonu: şehir başına 7 günlük hareketli ortalama
w7 = Window.partitionBy("city").orderBy("dt").rowsBetween(-6, 0)
daily_city = daily_city.withColumn("revenue_7d_avg", F.round(F.avg("revenue").over(w7), 2))

daily_city.filter(F.col("city") == "İstanbul").orderBy("dt").show(10)

In [ ]:
# Her şehirde en çok harcayan 3 müşteri — rank() penceresi
spend = completed.groupBy("user_id").agg(F.round(F.sum("amount"), 2).alias("total"))
w_rank = Window.partitionBy("city").orderBy(F.desc("total"))
top3 = (spend.join(users, "user_id")
        .withColumn("rnk", F.dense_rank().over(w_rank))
        .filter("rnk <= 3")
        .select("city", "rnk", "user_id", "plan", "total")
        .orderBy("city", "rnk"))
top3.show(12)

In [ ]:
# Pivot: plan × durum matrisi ve koşullu sütunlar
(orders.join(users, "user_id")
       .groupBy("plan").pivot("status", ["COMPLETED", "CANCELLED", "REFUNDED"]).count()
       .withColumn("iptal_orani_%", F.round(F.col("CANCELLED") * 100 /
                   (F.col("COMPLETED") + F.col("CANCELLED") + F.col("REFUNDED")), 2))
       .show())

# Karmaşık tipler: array + explode, struct, JSON
events = spark.createDataFrame(
    [("u1", '{"device":"ios","tags":["kampanya","mobil"]}'),
     ("u2", '{"device":"web","tags":["organik"]}')], ["user", "payload"])
js = StructType([StructField("device", StringType()), StructField("tags", ArrayType(StringType()))])
(events.withColumn("p", F.from_json("payload", js))
       .select("user", "p.device", F.explode("p.tags").alias("tag"))
       .show())

## 4. Spark SQL

In [ ]:
orders.createOrReplaceTempView("orders")
users.createOrReplaceTempView("users")

spark.sql('''
WITH user_stats AS (
  SELECT u.city, u.plan, o.user_id,
         COUNT(*)              AS n_orders,
         ROUND(SUM(o.amount),2) AS total
  FROM orders o JOIN users u USING (user_id)
  WHERE o.status = 'COMPLETED'
  GROUP BY u.city, u.plan, o.user_id
)
SELECT city, plan,
       COUNT(*)                                   AS customers,
       ROUND(AVG(total), 2)                       AS avg_spend,
       ROUND(PERCENTILE_APPROX(total, 0.5), 2)    AS median_spend,
       ROUND(PERCENTILE_APPROX(total, 0.95), 2)   AS p95_spend
FROM user_stats
GROUP BY city, plan
ORDER BY avg_spend DESC
LIMIT 10
''').show()

In [ ]:
# Spark 4.0+ : SQL pipe sözdizimi (|>) — sorguyu yukarıdan aşağı okunur hale getirir
if int(spark.version.split(".")[0]) >= 4:
    spark.sql('''
    FROM orders
    |> WHERE status = 'COMPLETED'
    |> AGGREGATE COUNT(*) AS n, CAST(SUM(amount) AS BIGINT) AS revenue GROUP BY dt
    |> ORDER BY revenue DESC
    |> LIMIT 5
    ''').show()
else:
    print("Pipe sözdizimi Spark 4.0+ gerektirir. Mevcut sürüm:", spark.version)

## 5. Catalyst Optimizer: Planı Okumak

- **PushedFilters**: filtre Parquet okuyucusuna indirildi → row group istatistikleriyle (min/max) gereksiz bloklar atlanır.
- **ReadSchema**: yalnızca gereken sütunlar okunur (column pruning).
- **PartitionFilters**: yalnızca ilgili dizinler (`dt=...`) listelenir (partition pruning).

In [ ]:
q = (spark.read.parquet(f"{BASE}/orders_parquet")
     .filter((F.col("amount") > 900) & (F.col("status") == "REFUNDED"))
     .select("order_id", "amount"))
print("== Parquet: predicate pushdown + column pruning ==")
grep_plan(q, "PushedFilters", "ReadSchema")

print("\n== Tam fiziksel plan ==")
q.explain("formatted")

In [ ]:
by_dt = spark.read.parquet(f"{BASE}/orders_by_dt")
one_day = by_dt.filter(F.col("dt") == "2026-09-15")

print("== Partition pruning ==")
grep_plan(one_day, "PartitionFilters")

with timer("Tüm veri (bölümsüz parquet), tek gün filtresi"):
    run(spark.read.parquet(f"{BASE}/orders_parquet").filter(F.col("dt") == "2026-09-15"))
with timer("dt ile bölümlenmiş parquet, tek gün filtresi"):
    run(one_day)

n_files = one_day.select(F.input_file_name()).distinct().count()
print("Okunan dosya sayısı:", n_files, "| toplam gün dizini:", len([d for d in os.listdir(f"{BASE}/orders_by_dt") if d.startswith("dt=")]))

## 6. Join Stratejileri, AQE ve Veri Çarpıklığı

In [ ]:
joined = orders.join(users, "user_id")

# 1) Otomatik broadcast kapalı → Sort-Merge Join (iki taraf da shuffle edilir)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
print("autoBroadcast KAPALI:"); grep_plan(joined, "Join", mode="simple")
with timer("Sort-Merge Join"):
    run(joined)

# 2) Broadcast ipucu → Broadcast Hash Join (büyük taraf shuffle edilmez)
bj = orders.join(F.broadcast(users), "user_id")
print("\nbroadcast() ipucu:"); grep_plan(bj, "Join", mode="simple")
with timer("Broadcast Hash Join"):
    run(bj)

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))   # varsayılana dön

### Veri çarpıklığı (skew) ve salting
`user_id`'ye göre hash ile bölümlediğimizde, çok sipariş veren birkaç kullanıcı aynı partition'a düşer ve o partition'ı işleyen task diğerlerinden çok daha uzun sürer (**straggler**).

In [ ]:
print("En çok sipariş veren 5 kullanıcı:")
orders.groupBy("user_id").count().orderBy(F.desc("count")).show(5)

def partition_sizes(df, n=8):
    rows = (df.groupBy(F.spark_partition_id().alias("pid")).count().orderBy("pid").collect())
    sizes = [r["count"] for r in rows]
    print("  partition boyutları:", sizes, f"| max/ortalama = {max(sizes) / (sum(sizes)/len(sizes)):.2f}")

print("user_id ile hash partition:")
partition_sizes(orders.repartition(8, "user_id"))

# Salting: anahtara rastgele bir 'tuz' ekleyerek sıcak anahtarı N parçaya böl
SALT = 8
salted = orders.withColumn("salt", (F.rand(11) * SALT).cast("int"))
print("(user_id, salt) ile hash partition:")
partition_sizes(salted.repartition(8, "user_id", "salt"))

# İki aşamalı toplama: önce (user_id, salt) ile kısmi toplam, sonra user_id ile nihai toplam
two_phase = (salted.groupBy("user_id", "salt").agg(F.sum("amount").alias("part"))
                   .groupBy("user_id").agg(F.round(F.sum("part"), 2).alias("total")))
direct = orders.groupBy("user_id").agg(F.round(F.sum("amount"), 2).alias("total"))
diff = two_phase.join(direct, "user_id").filter(F.abs(two_phase.total - direct.total) > 0.05).count()
print("Salting sonrası sonuçlar aynı mı?", diff == 0)

In [ ]:
# AQE ayarları — çalışma zamanında: partition birleştirme, skew join bölme, join stratejisi değiştirme
for k in ["spark.sql.adaptive.enabled",
          "spark.sql.adaptive.coalescePartitions.enabled",
          "spark.sql.adaptive.skewJoin.enabled",
          "spark.sql.adaptive.skewJoin.skewedPartitionFactor",
          "spark.sql.adaptive.advisoryPartitionSizeInBytes"]:
    print(f"{k:55s} = {spark.conf.get(k)}")

# AQE açıkken planın en üstünde 'AdaptiveSparkPlan isFinalPlan=...' görünür; çalıştıktan sonra true olur
agg = orders.groupBy("status").count()
agg.collect()          # aynı DataFrame çalıştıktan sonra plan "final" hale gelir
grep_plan(agg, "AdaptiveSparkPlan", "AQEShuffleRead", mode="simple")

## 7. Cache / Persist

In [ ]:
from pyspark import StorageLevel

enriched = (orders.join(F.broadcast(users), "user_id")
                  .withColumn("hour", F.hour("created_at"))
                  .filter("status = 'COMPLETED'"))

with timer("1. sorgu (cache yok)"):
    enriched.groupBy("city").count().collect()
with timer("2. sorgu (cache yok, her şey yeniden hesaplanır)"):
    enriched.groupBy("hour").count().collect()

enriched.persist(StorageLevel.MEMORY_AND_DISK)
with timer("cache'i doldur (count)"):
    enriched.count()
with timer("3. sorgu (cache'ten)"):
    enriched.groupBy("city").count().collect()
with timer("4. sorgu (cache'ten)"):
    enriched.groupBy("hour").count().collect()

print("Depolama seviyesi:", enriched.storageLevel)
print("👉 Spark UI → Storage sekmesinde önbellekteki boyutu görün.")
enriched.unpersist()

## 8. Python UDF vs Yerleşik Fonksiyon vs Pandas UDF
Python UDF'leri Catalyst için kara kutudur ve her satır JVM ↔ Python arasında serileştirilir. **Önce yerleşik fonksiyonları** deneyin; gerekiyorsa Apache Arrow tabanlı vektörize **Pandas UDF** kullanın.

In [ ]:
import pandas as pd
from pyspark.sql.functions import udf, pandas_udf

def segment_py(amount):
    return "yüksek" if amount >= 700 else ("orta" if amount >= 300 else "düşük")

segment_udf = udf(segment_py, StringType())

@pandas_udf(StringType())
def segment_pandas(a: pd.Series) -> pd.Series:
    return pd.cut(a, bins=[-1, 300, 700, float("inf")], right=False,
                  labels=["düşük", "orta", "yüksek"]).astype(str)

segment_builtin = (F.when(F.col("amount") >= 700, "yüksek")
                    .when(F.col("amount") >= 300, "orta").otherwise("düşük"))

# Disk okuma süresi ölçümü bozmasın diye girdiyi önce belleğe alıyoruz
amounts = orders.select("order_id", "amount").cache()
print("Satır:", amounts.count())

with timer("Python UDF"):
    run(amounts.withColumn("seg", segment_udf("amount")))
with timer("Pandas UDF (Arrow)"):
    run(amounts.withColumn("seg", segment_pandas("amount")))
with timer("Yerleşik when/otherwise"):
    run(amounts.withColumn("seg", segment_builtin))

# Üç yöntem aynı sonucu veriyor mu?
chk = (amounts.limit(50_000)
       .withColumn("a", segment_udf("amount"))
       .withColumn("b", segment_pandas("amount"))
       .withColumn("c", segment_builtin)
       .filter("a != b OR b != c").count())
print("Farklı sonuç veren satır:", chk)
amounts.unpersist()

## 9. Structured Streaming

### 9.1 `rate` kaynağı + olay zamanı penceresi + watermark
`rate` kaynağı saniyede belirli sayıda `(timestamp, value)` satırı üretir; bunu IoT sensör akışı gibi kullanacağız.

In [ ]:
stream = (spark.readStream.format("rate").option("rowsPerSecond", 200).load()
          .withColumn("device_id", F.concat(F.lit("sensor-"), (F.col("value") % 5).cast("string")))
          .withColumn("temp", F.round(20 + F.rand() * 15, 1))
          .withColumnRenamed("timestamp", "ts"))

windowed = (stream
            .withWatermark("ts", "10 seconds")                         # 10 sn'ye kadar geç veriyi kabul et
            .groupBy(F.window("ts", "5 seconds"), "device_id")         # 5 sn'lik düşen (tumbling) pencere
            .agg(F.count("*").alias("n"), F.round(F.avg("temp"), 2).alias("avg_temp"),
                 F.max("temp").alias("max_temp")))

q1 = (windowed.writeStream
      .outputMode("complete")            # tüm sonuç tablosunu her batch'te yeniden yaz (bellek sink'i için)
      .format("memory").queryName("iot_agg")
      .trigger(processingTime="2 seconds")
      .start())

time.sleep(15)
print("Durum:", q1.status["message"])
print("Son ilerleme: input satır =", q1.lastProgress["numInputRows"] if q1.lastProgress else None)
spark.sql("SELECT window.start, window.end, device_id, n, avg_temp, max_temp FROM iot_agg ORDER BY window.start DESC, device_id").show(10, truncate=False)
q1.stop()

### 9.2 Dosya kaynağı + `foreachBatch` + `trigger(availableNow=True)`
`availableNow`, o ana kadar biriken tüm veriyi (gerekirse birden çok micro-batch ile) işler ve durur — **zamanlanmış artımlı batch** işler için ideal. `checkpointLocation` hangi dosyaların işlendiğini hatırlar; ikinci çalıştırmada yalnızca **yeni** dosyalar işlenir.

In [ ]:
import json, shutil, glob

IN_DIR, OUT_DIR, CHK = f"{BASE}/stream_in", f"{BASE}/stream_out", f"{BASE}/_chk/files"
for d in (IN_DIR, OUT_DIR, CHK):
    shutil.rmtree(d, ignore_errors=True)
os.makedirs(IN_DIR)

def drop_files(start, n_files, rows=1000):
    for i in range(start, start + n_files):
        with open(f"{IN_DIR}/events_{i:03d}.json", "w") as f:
            for j in range(rows):
                f.write(json.dumps({"event_id": i * rows + j, "user_id": random.randint(1, 500),
                                    "type": random.choice(["view", "cart", "purchase"]),
                                    "price": round(random.uniform(5, 500), 2)}) + "\n")

ev_schema = "event_id LONG, user_id INT, type STRING, price DOUBLE"
batch_log = []

def process_batch(batch_df, batch_id):
    # foreachBatch içinde istediğiniz batch kodunu yazabilirsiniz: MERGE, birden çok hedefe yazma, JDBC...
    n = batch_df.count()
    (batch_df.groupBy("type").agg(F.count("*").alias("n"), F.round(F.sum("price"), 2).alias("tutar"))
             .withColumn("batch_id", F.lit(batch_id))
             .write.mode("append").parquet(OUT_DIR))
    batch_log.append((batch_id, n))

def run_incremental():
    q = (spark.readStream.schema(ev_schema).option("maxFilesPerTrigger", 2).json(IN_DIR)
         .writeStream.foreachBatch(process_batch)
         .option("checkpointLocation", CHK)
         .trigger(availableNow=True).start())
    q.awaitTermination()

drop_files(0, 5);  run_incremental(); print("1. çalıştırma batch'leri (batch_id, satır):", batch_log)
batch_log.clear()
drop_files(5, 2);  run_incremental(); print("2. çalıştırma — yalnızca yeni dosyalar:", batch_log)

spark.read.parquet(OUT_DIR).groupBy("type").agg(F.sum("n").alias("toplam_olay")).show()

## 10. MLlib: Churn (Müşteri Kaybı) Modeli
Sipariş verilerinden kullanıcı özellikleri çıkarıp, sentetik bir churn etiketiyle **Pipeline + CrossValidator** kuruyoruz.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

feats = (orders.groupBy("user_id")
         .agg(F.count("*").alias("n_orders"),
              F.avg(F.when(F.col("status") != "COMPLETED", 1.0).otherwise(0.0)).alias("problem_rate"),
              F.avg("amount").alias("avg_amount"),
              F.datediff(F.lit("2026-10-01"), F.max("dt")).alias("recency_days"))
         .join(users, "user_id")
         .withColumn("tenure_days", F.datediff(F.lit("2026-10-01"), "signup_date")))

# Sentetik etiket: az sipariş, uzun süredir alışveriş yapmama, sorunlu sipariş ve 'free' plan churn olasılığını artırır
logit = (-1.2 - 0.04 * F.col("n_orders") + 0.12 * F.col("recency_days") + 6.0 * F.col("problem_rate")
         + F.when(F.col("plan") == "free", 0.8).otherwise(0.0) + F.randn(21) * 0.8)
data = feats.withColumn("label", (1 / (1 + F.exp(-logit)) > 0.5).cast("double")).cache()
data.groupBy("label").count().show()

train, test = data.randomSplit([0.8, 0.2], seed=42)

idx = StringIndexer(inputCols=["plan", "city"], outputCols=["plan_i", "city_i"])
ohe = OneHotEncoder(inputCols=["plan_i", "city_i"], outputCols=["plan_v", "city_v"])
num_cols = ["n_orders", "problem_rate", "avg_amount", "recency_days", "tenure_days"]
vec = VectorAssembler(inputCols=num_cols + ["plan_v", "city_v"], outputCol="raw")
scl = StandardScaler(inputCol="raw", outputCol="features")
lr  = LogisticRegression(labelCol="label", featuresCol="features", maxIter=50)

pipe = Pipeline(stages=[idx, ohe, vec, scl, lr])
grid = ParamGridBuilder().addGrid(lr.regParam, [0.0, 0.05]).addGrid(lr.elasticNetParam, [0.0, 0.5]).build()
ev   = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
cv   = CrossValidator(estimator=pipe, estimatorParamMaps=grid, evaluator=ev, numFolds=3, parallelism=2, seed=7)

with timer("CrossValidator (4 kombinasyon × 3 kat)"):
    cv_model = cv.fit(train)

best = cv_model.bestModel
print(f"Lojistik regresyon test AUC: {ev.evaluate(best.transform(test)):.4f}")
print("En iyi regParam / elasticNet:", best.stages[-1].getRegParam(), best.stages[-1].getElasticNetParam())

coefs = best.stages[-1].coefficients.toArray()[:len(num_cols)]
for c, v in sorted(zip(num_cols, coefs), key=lambda t: -abs(t[1])):
    print(f"  {c:14s} {v:+.3f}")

In [ ]:
# Aynı ön işleme hattıyla Random Forest — özellik önemleri
rf = RandomForestClassifier(labelCol="label", featuresCol="raw", numTrees=40, maxDepth=6, seed=3)
rf_model = Pipeline(stages=[idx, ohe, vec, rf]).fit(train)
print(f"Random Forest test AUC: {ev.evaluate(rf_model.transform(test)):.4f}")

imp = rf_model.stages[-1].featureImportances.toArray()[:len(num_cols)]
for c, v in sorted(zip(num_cols, imp), key=lambda t: -t[1]):
    print(f"  {c:14s} {'█' * int(v * 60)} {v:.3f}")

# Modeli kaydet / yükle
path = f"{BASE}/models/churn_lr"
best.write().overwrite().save(path)
from pyspark.ml import PipelineModel
print("Kaydedilen model yüklendi, aşama sayısı:", len(PipelineModel.load(path).stages))
data.unpersist()

## 11. Executor Boyutlandırma Hesaplayıcısı (YARN kümesi için)

In [ ]:
import math

def size_executors(nodes, cores_per_node, ram_gb_per_node, cores_per_executor=5,
                   overhead_ratio=0.10, input_gb=500, target_partition_mb=128):
    usable_cores = cores_per_node - 1            # OS ve Hadoop daemon'ları için 1 çekirdek
    usable_ram   = ram_gb_per_node - 1           # OS için 1 GB
    execs_per_node = max(1, usable_cores // cores_per_executor)
    num_executors  = execs_per_node * nodes - 1  # 1 executor yeri ApplicationMaster/driver için
    mem_per_exec   = usable_ram / execs_per_node
    heap_gb        = math.floor(mem_per_exec / (1 + overhead_ratio))
    overhead_gb    = max(0.384, mem_per_exec - heap_gb)
    parallel_tasks = num_executors * cores_per_executor
    parts = max(math.ceil(input_gb * 1024 / target_partition_mb), parallel_tasks * 3)
    parts = math.ceil(parts / parallel_tasks) * parallel_tasks   # çekirdek sayısının katına yuvarla
    return {
        "--num-executors": num_executors,
        "--executor-cores": cores_per_executor,
        "--executor-memory": f"{heap_gb}g",
        "spark.executor.memoryOverhead": f"{overhead_gb:.1f}g",
        "paralel task kapasitesi": parallel_tasks,
        "spark.sql.shuffle.partitions": parts,
    }

for k, v in size_executors(nodes=10, cores_per_node=32, ram_gb_per_node=256, input_gb=500).items():
    print(f"{k:32s} {v}")

## 12. Alıştırmalar 🏋️

1. **Lineage:** Bölüm 1'deki `counts` RDD'sine bir `filter` ve bir `sortByKey` ekleyin. `toDebugString()` çıktısında kaç stage oluştu? Neden?
2. **Partition ayarı:** `spark.sql.shuffle.partitions` değerini 2, 8 ve 200 yaparak Bölüm 3'teki `daily_city` sorgusunu `run()` ile ölçün. AQE'yi kapatıp (`spark.sql.adaptive.enabled=false`) tekrarlayın. Ne değişti?
3. **Pushdown'ı bozun:** Bölüm 5'teki filtreyi bir Python UDF içine taşıyın (`udf(lambda a: a > 900)`). `PushedFilters` satırında ne oldu?
4. **Skew join:** `orders` ile en çok sipariş veren 100 kullanıcıyı içeren bir tabloyu, broadcast'i kapatarak join edin. Spark UI'da task süresi dağılımına bakın; sonra salting ile düzeltin.
5. **Streaming:** Bölüm 9.1'deki sorguyu `outputMode("append")` ile bir parquet dizinine yazın. Pencereler ne zaman diske düşüyor? Watermark'ı 30 saniyeye çıkarınca ne olur?
6. **MLlib:** `GBTClassifier` ekleyip AUC'yi lojistik regresyon ve Random Forest ile karşılaştırın.
7. **Hadoop köprüsü:** Bölüm 1 not defterindeki HDFS kümesini çalıştırıp `spark.read.text("hdfs://localhost:9000/...")` ile HDFS'ten okuyun.

In [ ]:
# Laboratuvarı kapat
spark.stop()
print("SparkSession kapatıldı.")